In [1]:
!pip install -q transformers accelerate peft trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
import os
from google.colab import files
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import DPOConfig, DPOTrainer
from datasets import Dataset
from huggingface_hub import notebook_login

print("Imports loaded!")

Imports loaded!


In [3]:
!pip install -U torchao>=0.16.0

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"\n⏳ Loading {MODEL_NAME}...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

print("✅ Base model loaded!")

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(" LoRA config ready!")


⏳ Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Base model loaded!
 LoRA config ready!


In [14]:
from trl import DPOConfig, DPOTrainer
from datasets import Dataset
from google.colab import files
import os
import zipfile

print("="*50)
print("DPO PHASE 1: Training on samples 0-990")
print("="*50)

# Format DPO data
def format_dpo(row):
    prompt = f"""Sort the following objects from BIGGEST to SMALLEST.

Objects: {row['sample']}

OUTPUT:"""

    return {
        "prompt": prompt,
        "chosen": row['cot'],
        "rejected": row['reject']
    }

dpo_data = training_data.apply(format_dpo, axis=1)
dpo_list = dpo_data.tolist()
dpo_dataset = Dataset.from_list(dpo_list)

dpo_phase1 = dpo_dataset.select(range(990))
print(f" DPO Phase 1 data: {len(dpo_phase1)} samples")

training_args = DPOConfig(
    output_dir="./dpo_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=10,
    optim="paged_adamw_8bit",
    report_to="none",
    beta=0.1,
    bf16=False,
    fp16=True,
)

trainer = DPOTrainer(
    model=base_model,
    ref_model=None,
    args=training_args,
    train_dataset=dpo_phase1,
    peft_config=lora_config,
)

print("DPO Phase 1 Training (990 samples, 2 epochs)...")
trainer.train()
print(" DPO Phase 1 complete!")

trainer.save_model("./dpo_adapter")
print(" DPO adapter saved!")

# Download
if os.path.exists("./dpo_adapter"):
    with zipfile.ZipFile("dpo_adapter_phase1.zip", 'w') as zipf:
        for root, dirs, files_list in os.walk("./dpo_adapter"):
            for file in files_list:
                zipf.write(os.path.join(root, file),
                          os.path.relpath(os.path.join(root, file), "./dpo_adapter"))
    files.download("dpo_adapter_phase1.zip")
    print(" dpo_adapter_phase1.zip downloaded!")

print("DPO PHASE 1 COMPLETE!")

DPO PHASE 1: Training on samples 0-990
 DPO Phase 1 data: 990 samples


Adding EOS to train dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


...

Step,Training Loss
10,0.367990
20,0.094433
30,0.020687
40,0.009845
50,0.004722
60,0.020823
70,0.050035
80,0.008932
90,0.003544
100,0.001111


 DPO Phase 1 complete!
 DPO adapter saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 dpo_adapter_phase1.zip downloaded!
DPO PHASE 1 COMPLETE!


In [16]:
if os.path.exists("./dpo_adapter"):
    with zipfile.ZipFile("dpo_adapter_phase1.zip", 'w') as zipf:
        for root, dirs, files_list in os.walk("./dpo_adapter"):
            for file in files_list:
                zipf.write(os.path.join(root, file),
                          os.path.relpath(os.path.join(root, file), "./dpo_adapter"))
    files.download("dpo_adapter_phase1.zip")
    print(" dpo_adapter_phase1.zip downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 dpo_adapter_phase1.zip downloaded!


In [18]:
from trl import DPOConfig, DPOTrainer
from datasets import Dataset
from peft import PeftModel
from google.colab import files
import os
import zipfile

print("="*50)
print("DPO PHASE 2: Training on samples 990-1980")
print("="*50)

# Load Phase 1 adapter
model = PeftModel.from_pretrained(base_model, "./dpo_adapter")
print(" Phase 1 adapter loaded!")


dpo_phase2 = dpo_dataset.select(range(990, 1980))
print(f" DPO Phase 2 data: {len(dpo_phase2)} samples")

training_args = DPOConfig(
    output_dir="./dpo_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=10,
    optim="paged_adamw_8bit",
    report_to="none",
    beta=0.1,
    bf16=False,
    fp16=True,
)

trainer = DPOTrainer(
    model=base_model,
    ref_model=None,
    args=training_args,
    train_dataset=dpo_phase2,
    peft_config=lora_config,
)

print("⏳ DPO Phase 2 Training (990 samples, 2 epochs)...")
trainer.train()
print(" DPO Phase 2 complete!")

trainer.save_model("./dpo_adapter")
print(" DPO adapter saved! (Phase 1 + Phase 2 combined)")

# Download final adapter
if os.path.exists("./dpo_adapter"):
    with zipfile.ZipFile("dpo_adapter_final.zip", 'w') as zipf:
        for root, dirs, files_list in os.walk("./dpo_adapter"):
            for file in files_list:
                zipf.write(os.path.join(root, file),
                          os.path.relpath(os.path.join(root, file), "./dpo_adapter"))
    files.download("dpo_adapter_final.zip")
    print("dpo_adapter_final.zip downloaded!")



DPO PHASE 2: Training on samples 990-1980
 Phase 1 adapter loaded!
 DPO Phase 2 data: 990 samples


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

...

Step,Training Loss
10,0.308564
20,0.083575
30,0.047385
40,0.072747
50,0.018139
60,0.033316
70,0.002838
80,0.005425
90,0.001860
100,0.008866


 DPO Phase 2 complete!
 DPO adapter saved! (Phase 1 + Phase 2 combined)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

dpo_adapter_final.zip downloaded!


In [19]:
from peft import PeftModel
from transformers import pipeline, GenerationConfig
from transformers import set_seed
import pandas as pd

print("="*50)
print("EVALUATING DPO MODEL (Part 2)")
print("="*50)

# Load DPO adapter
dpo_model = PeftModel.from_pretrained(base_model, "./dpo_adapter")
dpo_model = dpo_model.merge_and_unload()
dpo_model.eval()

generator = pipeline(
    task="text-generation",
    model=dpo_model,
    tokenizer=tokenizer,
    clean_up_tokenization_spaces=False,
)

def generate_response(prompt, generator, seed=42):
    gen_config = GenerationConfig(
        temperature=0.1,
        do_sample=True,
        max_new_tokens=256,
    )
    set_seed(seed)
    messages = [{"role": "user", "content": prompt}]
    outputs = generator(messages, generation_config=gen_config)

    full_output = outputs[0]['generated_text']
    if isinstance(full_output, list):
        for msg in reversed(full_output):
            if msg.get('role') == 'assistant':
                response_text = msg.get('content', '')
                if "OUTPUT:" in response_text:
                    return "OUTPUT: " + response_text.split("OUTPUT:")[-1].strip()
                return response_text.strip()
    if isinstance(full_output, str):
        if "OUTPUT:" in full_output:
            return "OUTPUT: " + full_output.split("OUTPUT:")[-1].strip()
        return full_output.strip()
    return str(full_output)

results = []

for idx, row in testing_data.iterrows():
    prompt = f"""Sort the following objects from BIGGEST to SMALLEST.

Objects: {row['sample']}

OUTPUT:"""

    output = generate_response(prompt, generator)

    results.append({
        'sample_index': idx,
        'sample': row['sample'],
        'llm_output': output
    })

results_df = pd.DataFrame(results)
results_df.to_csv("part2.csv", index=False)

print(" Part 2 results saved to part2.csv")
print("\n📋 First 3 results:")
print(results_df.head(3))
print("\n PART 2 (DPO) COMPLETE!")

files.download("part2.csv")

EVALUATING DPO MODEL (Part 2)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

 Part 2 results saved to part2.csv

📋 First 3 results:
   sample_index                                             sample  \
0             0     paperclip, carbon atom, mouse, toaster, saturn   
1             1        lion, galaxy, coin, skyscraper, carbon atom   
2             2  skyscraper, continent, grain of sand, city, horse   

                                          llm_output  
0  1. Paperclip, Carbon atom, Mouse, Toaster, Sat...  
1  1. Galaxy\n2. Coin\n3. Skyscraper\n4. Carbon a...  
2  1. Skyscraper, continent, grain of sand, city,...  

 PART 2 (DPO) COMPLETE!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>